# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (tables), fields, and their IDs (`@id`).

In [ ]:
# List all record sets and their fields
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'record_set'):
    record_sets = metadata.record_set
if not record_sets or len(record_sets) == 0:
    # Try to extract record set IDs from distributions
    print("No explicit record sets found in the Croissant metadata. Listing data files (distributions):")
    if hasattr(metadata, 'distribution'):
        distributions = metadata.distribution
        for dist in distributions:
            print(f"Distribution @id: {getattr(dist, '@id', None)}; name: {getattr(dist, 'name', None)}; contentUrl: {getattr(dist, 'content_url', None)}")
    else:
        print("No distributions found either.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)}")
        print("  Fields:")
        fields = getattr(rs, 'field', None) if hasattr(rs, 'field') else rs.get('field', []) if isinstance(rs, dict) else []
        if fields:
            for field in fields:
                fid = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', None)
                fname = field['name'] if isinstance(field, dict) and 'name' in field else getattr(field, 'name', None)
                print(f"    - @id: {fid}, name: {fname}")
        else:
            print("    (No fields found)")

### View Sample Records
Since the Croissant metadata does not declare explicit `recordSet` entries, we will list and preview records from any available distribution data files. If there are tables, they will show here.

In [ ]:
# List/preview records from main dataset
# If there is a distribution listed, try using its @id as the record_set argument
main_dist_id = None
if hasattr(metadata, 'distribution'):
    dists = metadata.distribution
    if dists and len(dists) > 0:
        # Pick the first one as main
        main_dist_id = getattr(dists[0], '@id', None)

if main_dist_id:
    try:
        print(f"Previewing first 2 records from distribution/record set: {main_dist_id}")
        for i, record in enumerate(dataset.records(record_set=main_dist_id)):
            print(json.dumps(record, indent=2))
            if i >= 1:
                break
    except Exception as e:
        print(f"Could not iterate dataset.records for {main_dist_id}: {e}")
else:
    print("No distribution available to load records.")

## 3. Data Extraction
Load data from the main record set (distribution) into a DataFrame for analysis. All entities are referenced by their `@id`.

In [ ]:
# Extract data from distribution (main data file)
# Use the distribution's @id as the record_set identifier
record_sets_ids = []
if main_dist_id:
    record_sets_ids = [main_dist_id]
else:
    print('No record set or distribution found.')

dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load data for record_set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping by key attributes. Examples below demonstrate numeric field selection, filtering, and grouping.

In [ ]:
# Specify the main DataFrame variable
import warnings
warnings.filterwarnings('ignore')

if len(record_sets_ids) > 0:
    record_set_id = record_sets_ids[0]
    df = dataframes[record_set_id]
    # Try to detect a numeric field for demonstration (e.g., MW, coverage, peptide_counts)
    numeric_field = None
    for col in df.columns:
        if 'mw' in col.lower() or 'weight' in col.lower() or 'coverage' in col.lower():
            numeric_field = col
            break
    if not numeric_field:
        # Otherwise, pick the first numeric column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    if numeric_field:
        print(f"Analyzing numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        try:
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
            print(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a likely categorical field (e.g., description, accession, or other string fields)
            group_field = None
            # Prefer to not use identifiers; search for 'description', 'type', or other
            for col in df.columns:
                if col != numeric_field and (df[col].dtype == object and df[col].nunique() < df.shape[0] and df[col].nunique() > 1):
                    group_field = col
                    break
            if group_field is not None:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
                print(grouped_df.head())
        except Exception as e:
            print(f"Error during EDA: {e}")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field. If two suitable numeric columns are present, visualize their relationship.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets_ids) > 0 and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot if another numeric field exists
    second_numeric = None
    for col in df.columns:
        if col != numeric_field and pd.api.types.is_numeric_dtype(df[col]):
            second_numeric = col
            break
    if second_numeric:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[numeric_field], y=df[second_numeric])
        plt.xlabel(numeric_field)
        plt.ylabel(second_numeric)
        plt.title(f"Scatter plot: {numeric_field} vs {second_numeric}")
        plt.show()
else:
    print('No numeric data available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to explore the FAIR² Croissant dataset describing mass spectrometry analysis of proteins in extracellular vesicles from stimulated human mast cells.

- We loaded the Croissant metadata using only the URL as input.
- We discovered the available distribution(s) (data files) and examined example records.
- We dynamically selected numeric fields for exploratory analysis and basic statistical processing.
- We visualized record distributions and relationships with common Python data tools.

**Further analysis** can include more advanced filtering (e.g., by different protein types), join or merge operations with external annotation tables, and domain-specific visualization. Remember to reference each field, table, or column by their Croissant `@id` for optimal FAIR and programmatic reproducibility.